# 0.3 Bandwidth Content: Access Lists, Authorization Lists, And Blob Hashes

This notebook builds the block-level bandwidth content table after calldata and BAL estimation already exist.

It adds the transaction-content byte components that are not covered by calldata + BAL:

- EIP-2930/EIP-7981 transaction access-list entries, fetched from JSON-RPC full transaction objects.
- EIP-7702 authorization tuples, fetched from JSON-RPC type-4 transactions.
- EIP-4844 blob versioned hashes, counted from Xatu `execution_transaction.blob_hashes`.

For this notebook, we only produce bandwidth-facing byte counts. Gas pricing for these bytes is intentionally left to the replay mechanism that consumes the table.

In [1]:
import os
from pathlib import Path

import clickhouse_connect
import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from sim.rpc_access_lists import fetch_access_list_data_for_blocks
from sim.rpc_authorizations import fetch_authorization_data_for_blocks
from sim.xatu_calldata import query_xatu_calldata_by_block

pd.options.display.float_format = "{:,.4f}".format

load_dotenv(PROJECT_ROOT / ".env")
missing = [name for name in ["CLICKHOUSE_USER", "CLICKHOUSE_PASSWORD"] if not os.environ.get(name)]
if missing:
    raise RuntimeError("Missing .env values: " + ", ".join(missing))

CLICKHOUSE_RAW_HOST = os.environ.get("CLICKHOUSE_RAW_HOST", "clickhouse-raw.xatu.ethpandaops.io")
raw_client = clickhouse_connect.get_client(
    host=CLICKHOUSE_RAW_HOST,
    port=443,
    secure=True,
    username=os.environ["CLICKHOUSE_USER"],
    password=os.environ["CLICKHOUSE_PASSWORD"],
)

ETHNODEOPS_API_KEY = os.environ.get("ETHNODEOPS_API_KEY") or os.environ.get("hoodi_api_key")
ETHNODEOPS_RPC = os.environ.get("ETHNODEOPS_RPC", "https://erigon.mainnet.rpc.ethnodeops.xyz")
ALCHEMY_RPC = os.environ.get("ALCHEMY_RPC")

if ETHNODEOPS_API_KEY:
    RPC_URL = ETHNODEOPS_RPC
    RPC_HEADERS = {"X-API-Key": ETHNODEOPS_API_KEY}
    RPC_PROVIDER_LABEL = "ethnodeops_erigon_mainnet" if "erigon." in RPC_URL else "ethnodeops_mainnet"
elif ALCHEMY_RPC:
    RPC_URL = ALCHEMY_RPC
    RPC_HEADERS = None
    RPC_PROVIDER_LABEL = "alchemy_mainnet"
else:
    raise RuntimeError("Missing ETHNODEOPS_API_KEY or ALCHEMY_RPC in .env")

print("raw", CLICKHOUSE_RAW_HOST, raw_client.query("SELECT version()").result_rows)
print("rpc_provider", RPC_PROVIDER_LABEL)

raw clickhouse-raw.xatu.ethpandaops.io [('26.2.5.45',)]
rpc_provider ethnodeops_erigon_mainnet


## Parameters

In [2]:
NETWORK = "mainnet"
START_BLOCK = 24_120_001
N_BLOCKS = 500
BLOCKS = list(range(START_BLOCK, START_BLOCK + N_BLOCKS))
RPC_MAX_WORKERS = 32
RPC_MAX_RETRIES = 6
RPC_RETRY_SLEEP_SECONDS = 1.5

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

AUTH_RECORDS_CSV = DATA_DIR / f"rpc_authorization_records_{min(BLOCKS)}_{max(BLOCKS)}.csv"
AUTH_SUMMARY_CSV = DATA_DIR / f"rpc_authorization_summary_{min(BLOCKS)}_{max(BLOCKS)}.csv"
ACCESS_LIST_RECORDS_CSV = DATA_DIR / f"rpc_access_list_records_{min(BLOCKS)}_{max(BLOCKS)}.csv"
ACCESS_LIST_SUMMARY_CSV = DATA_DIR / f"rpc_access_list_summary_{min(BLOCKS)}_{max(BLOCKS)}.csv"
CALLDATA_CSV = DATA_DIR / f"xatu_calldata_{min(BLOCKS)}_{max(BLOCKS)}.csv"
BAL_CSV = DATA_DIR / f"rpc_bal_summary_{min(BLOCKS)}_{max(BLOCKS)}.csv"
CONTENT_CSV = DATA_DIR / f"bandwidth_content_8131_{min(BLOCKS)}_{max(BLOCKS)}.csv"

WRITE_CSV = True
min(BLOCKS), max(BLOCKS), len(BLOCKS), RPC_MAX_WORKERS

(24120001, 24120500, 500, 32)

## Pull Authorization Lists

The helper first queries Xatu for type-4 transaction hashes, then fetches each raw transaction from RPC and decodes the full `authorizationList`. Bandwidth counts every authorization tuple, including invalid or duplicate tuples, because they are bytes on the wire.

This notebook stores tuple counts plus both observed RLP bytes and the fixed-size EIP-8131 byte convention. It does not assign gas to authorization tuples.

In [3]:
if AUTH_RECORDS_CSV.exists() and AUTH_SUMMARY_CSV.exists():
    auth_records = pd.read_csv(AUTH_RECORDS_CSV)
    auth_summary = pd.read_csv(AUTH_SUMMARY_CSV)
    print("loaded", AUTH_RECORDS_CSV)
    print("loaded", AUTH_SUMMARY_CSV)
else:
    auth_records, auth_summary = fetch_authorization_data_for_blocks(
        raw_client=raw_client,
        rpc_url=RPC_URL,
        block_numbers=BLOCKS,
        network=NETWORK,
        rpc_headers=RPC_HEADERS,
        max_workers=RPC_MAX_WORKERS,
    )

auth_summary = auth_summary.drop(columns=["authorization_tuple_gas"], errors="ignore")

if WRITE_CSV:
    auth_records.to_csv(AUTH_RECORDS_CSV, index=False)
    auth_summary.to_csv(AUTH_SUMMARY_CSV, index=False)
    print(AUTH_RECORDS_CSV)
    print(AUTH_SUMMARY_CSV)

auth_summary.head()

loaded /Users/william/PycharmProjects/eip-7999-research/data/rpc_authorization_records_24120001_24120500.csv
loaded /Users/william/PycharmProjects/eip-7999-research/data/rpc_authorization_summary_24120001_24120500.csv
/Users/william/PycharmProjects/eip-7999-research/data/rpc_authorization_records_24120001_24120500.csv
/Users/william/PycharmProjects/eip-7999-research/data/rpc_authorization_summary_24120001_24120500.csv


,block_number,type4_tx_count,authorization_tuple_count,authorization_tuple_rlp_bytes,authorization_tuple_8131_bytes,authorization_set_tuple_count,authorization_clear_tuple_count,authorization_recovered_count,authorization_state_upper_bound_authorities
0,24120001,1,1,92,108,1,0,1,1
1,24120002,3,3,276,324,3,0,3,3
2,24120003,1,1,92,108,1,0,1,1
3,24120004,1,1,92,108,1,0,1,1
4,24120005,2,2,184,216,2,0,2,2


## Pull Transaction Access Lists

Xatu does not currently expose transaction access-list entries in the raw or canonical transaction tables. The notebook therefore uses RPC full transaction objects and summarizes the EIP-2930 access list per block.

For byte accounting, access-list bytes are `20 * address_entries + 32 * storage_keys`. The notebook does not assign gas to those bytes.

In [4]:
if ACCESS_LIST_RECORDS_CSV.exists() and ACCESS_LIST_SUMMARY_CSV.exists():
    access_list_records = pd.read_csv(ACCESS_LIST_RECORDS_CSV)
    access_list_summary = pd.read_csv(ACCESS_LIST_SUMMARY_CSV)
    print("loaded", ACCESS_LIST_RECORDS_CSV)
    print("loaded", ACCESS_LIST_SUMMARY_CSV)
else:
    access_list_records, access_list_summary = fetch_access_list_data_for_blocks(
        rpc_url=RPC_URL,
        block_numbers=BLOCKS,
        rpc_headers=RPC_HEADERS,
        max_retries=RPC_MAX_RETRIES,
        retry_sleep_seconds=RPC_RETRY_SLEEP_SECONDS,
        max_workers=RPC_MAX_WORKERS,
    )

access_list_records = access_list_records.drop(columns=["tx_access_list_gas"], errors="ignore")
access_list_summary = access_list_summary.drop(columns=["tx_access_list_gas"], errors="ignore")

if WRITE_CSV:
    access_list_records.to_csv(ACCESS_LIST_RECORDS_CSV, index=False)
    access_list_summary.to_csv(ACCESS_LIST_SUMMARY_CSV, index=False)
    print(ACCESS_LIST_RECORDS_CSV)
    print(ACCESS_LIST_SUMMARY_CSV)

access_list_summary.head()

loaded /Users/william/PycharmProjects/eip-7999-research/data/rpc_access_list_records_24120001_24120500.csv
loaded /Users/william/PycharmProjects/eip-7999-research/data/rpc_access_list_summary_24120001_24120500.csv


/Users/william/PycharmProjects/eip-7999-research/data/rpc_access_list_records_24120001_24120500.csv
/Users/william/PycharmProjects/eip-7999-research/data/rpc_access_list_summary_24120001_24120500.csv


,block_number,tx_access_list_tx_count,tx_access_list_address_count,tx_access_list_storage_key_count,tx_access_list_bytes
0,24120001,6,41,202,7284
1,24120002,0,0,0,0
2,24120003,2,10,24,968
3,24120004,5,27,104,3868
4,24120005,1,16,71,2592


## Pull Calldata And Blob Hash Counts

Calldata bytes come from Xatu. Blob versioned hashes are counted from Xatu `execution_transaction.blob_hashes`.

This notebook keeps byte counts only: zero/nonzero calldata bytes, total calldata bytes, blob-hash count, and blob-hash bytes.

In [5]:
if CALLDATA_CSV.exists():
    calldata = pd.read_csv(CALLDATA_CSV)
    print("loaded", CALLDATA_CSV)
else:
    calldata = query_xatu_calldata_by_block(raw_client, BLOCKS, network=NETWORK)

calldata = calldata.drop(
    columns=["calldata_gas", "calldata_gas_source", "blob_versioned_hash_gas"],
    errors="ignore",
)

if WRITE_CSV:
    calldata.to_csv(CALLDATA_CSV, index=False)
    print(CALLDATA_CSV)

calldata[[
    "block_number",
    "calldata_zero_bytes",
    "calldata_nonzero_bytes",
    "calldata_bytes",
    "blob_versioned_hash_count",
    "blob_versioned_hash_bytes",
]].head()

loaded /Users/william/PycharmProjects/eip-7999-research/data/xatu_calldata_24120001_24120500.csv
/Users/william/PycharmProjects/eip-7999-research/data/xatu_calldata_24120001_24120500.csv


,block_number,calldata_zero_bytes,calldata_nonzero_bytes,calldata_bytes,blob_versioned_hash_count,blob_versioned_hash_bytes
0,24120001,144495,94030,238525,13,416
1,24120002,29852,14640,44492,6,192
2,24120003,86377,50190,136567,4,128
3,24120004,44058,26402,70460,9,288
4,24120005,30210,16786,46996,0,0


## Load Existing BAL Estimates

This notebook expects the RPC BAL notebook to have already written `rpc_bal_summary_{start}_{end}.csv`. The join uses raw RLP BAL bytes with reads and system changes, matching the current BAL estimation path.

In [6]:
if not BAL_CSV.exists():
    raise FileNotFoundError(
        f"Missing {BAL_CSV}. Run notebooks/0.1-rpc-bal-rlp.ipynb for the same block range first."
    )

bal = pd.read_csv(BAL_CSV)
bal = bal[["block_number", "bal_rlp_bytes"]].copy()
bal.head()

,block_number,bal_rlp_bytes
0,24120001,339327
1,24120002,72309
2,24120003,186956
3,24120004,117839
4,24120005,81624


## Join Bandwidth Content

This table joins the bandwidth components currently available and keeps only byte fields plus the counts needed to understand fixed-size components.

```text
bandwidth_payload_bytes = calldata bytes + BAL RLP bytes + access-list bytes + actual authorization RLP bytes + blob versioned hash bytes
bandwidth_metered_bytes = calldata bytes + BAL RLP bytes + access-list bytes + EIP-8131 authorization bytes + blob versioned hash bytes
```

For transaction access lists, `tx_access_list_bytes` follows EIP-7981's `20 * addresses + 32 * storage_keys` byte count. For authorization tuples, `authorization_tuple_rlp_bytes` is the actual encoded payload size, while `authorization_tuple_8131_bytes` is the EIP-8131 fixed-size byte convention.

In [7]:
content = calldata.merge(bal, on="block_number", how="left")
content = content.merge(access_list_summary, on="block_number", how="left")
content = content.merge(auth_summary, on="block_number", how="left")

content = content.drop(
    columns=[
        "calldata_gas",
        "calldata_gas_source",
        "bal_gas",
        "tx_access_list_gas",
        "authorization_tuple_gas",
        "blob_versioned_hash_gas",
        "bandwidth_gas",
    ],
    errors="ignore",
)

fill_zero_cols = [
    "bal_rlp_bytes",
    "tx_access_list_tx_count",
    "tx_access_list_address_count",
    "tx_access_list_storage_key_count",
    "tx_access_list_bytes",
    "type4_tx_count",
    "authorization_tuple_count",
    "authorization_tuple_rlp_bytes",
    "authorization_tuple_8131_bytes",
    "blob_versioned_hash_count",
    "blob_versioned_hash_bytes",
]
for column in fill_zero_cols:
    if column in content.columns:
        content[column] = content[column].fillna(0).astype("int64")

content["bandwidth_payload_bytes"] = (
    content["calldata_bytes"]
    + content["bal_rlp_bytes"]
    + content["tx_access_list_bytes"]
    + content["authorization_tuple_rlp_bytes"]
    + content["blob_versioned_hash_bytes"]
)
content["bandwidth_metered_bytes"] = (
    content["calldata_bytes"]
    + content["bal_rlp_bytes"]
    + content["tx_access_list_bytes"]
    + content["authorization_tuple_8131_bytes"]
    + content["blob_versioned_hash_bytes"]
)

output_cols = [
    "block_number",
    "calldata_zero_bytes",
    "calldata_nonzero_bytes",
    "calldata_bytes",
    "bal_rlp_bytes",
    "tx_access_list_tx_count",
    "tx_access_list_address_count",
    "tx_access_list_storage_key_count",
    "tx_access_list_bytes",
    "authorization_tuple_count",
    "authorization_tuple_rlp_bytes",
    "authorization_tuple_8131_bytes",
    "blob_versioned_hash_count",
    "blob_versioned_hash_bytes",
    "bandwidth_payload_bytes",
    "bandwidth_metered_bytes",
]
content = content[output_cols].sort_values("block_number").reset_index(drop=True)

if WRITE_CSV:
    content.to_csv(CONTENT_CSV, index=False)
    print(CONTENT_CSV)

content.head()

/Users/william/PycharmProjects/eip-7999-research/data/bandwidth_content_8131_24120001_24120500.csv


,block_number,calldata_zero_bytes,calldata_nonzero_bytes,calldata_bytes,bal_rlp_bytes,tx_access_list_tx_count,tx_access_list_address_count,tx_access_list_storage_key_count,tx_access_list_bytes,authorization_tuple_count,authorization_tuple_rlp_bytes,authorization_tuple_8131_bytes,blob_versioned_hash_count,blob_versioned_hash_bytes,bandwidth_payload_bytes,bandwidth_metered_bytes
0,24120001,144495,94030,238525,339327,6,41,202,7284,1,92,108,13,416,585644,585660
1,24120002,29852,14640,44492,72309,0,0,0,0,3,276,324,6,192,117269,117317
2,24120003,86377,50190,136567,186956,2,10,24,968,1,92,108,4,128,324711,324727
3,24120004,44058,26402,70460,117839,5,27,104,3868,1,92,108,9,288,192547,192563
4,24120005,30210,16786,46996,81624,1,16,71,2592,2,184,216,0,0,131396,131428


## Summary

In [8]:
summary = pd.DataFrame(
    [
        {
            "blocks": len(content),
            "calldata_bytes": int(content["calldata_bytes"].sum()),
            "bal_rlp_bytes": int(content["bal_rlp_bytes"].sum()),
            "tx_access_list_tx_count": int(content["tx_access_list_tx_count"].sum()),
            "tx_access_list_address_count": int(content["tx_access_list_address_count"].sum()),
            "tx_access_list_storage_key_count": int(content["tx_access_list_storage_key_count"].sum()),
            "tx_access_list_bytes": int(content["tx_access_list_bytes"].sum()),
            "authorization_tuple_count": int(content["authorization_tuple_count"].sum()),
            "authorization_tuple_rlp_bytes": int(content["authorization_tuple_rlp_bytes"].sum()),
            "authorization_tuple_8131_bytes": int(content["authorization_tuple_8131_bytes"].sum()),
            "blob_versioned_hash_count": int(content["blob_versioned_hash_count"].sum()),
            "blob_versioned_hash_bytes": int(content["blob_versioned_hash_bytes"].sum()),
            "bandwidth_payload_bytes": int(content["bandwidth_payload_bytes"].sum()),
            "bandwidth_metered_bytes": int(content["bandwidth_metered_bytes"].sum()),
        }
    ]
)
summary

,blocks,calldata_bytes,bal_rlp_bytes,tx_access_list_tx_count,tx_access_list_address_count,tx_access_list_storage_key_count,tx_access_list_bytes,authorization_tuple_count,authorization_tuple_rlp_bytes,authorization_tuple_8131_bytes,blob_versioned_hash_count,blob_versioned_hash_bytes,bandwidth_payload_bytes,bandwidth_metered_bytes
0,500,57151077,74084091,2236,19158,84529,3088088,637,59061,68796,2174,69568,134451885,134461620
